# FMCG Demand Forecasting - Synthetic EDA Demo

This notebook is safe for public GitHub because it uses `data/sample/synthetic_sales.csv`, not confidential company data.

Purpose:
- Inspect weekly demand and CBM structure.
- Show simple business EDA for FMCG supply chain planning.
- Keep model training in `src/train_model.py` so the main pipeline stays reproducible.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT_DIR = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sample_path = ROOT_DIR / "data" / "sample" / "synthetic_sales.csv"

sales = pd.read_csv(sample_path, parse_dates=["ACTUALSHIPDATE"])
sales.head()

## Dataset Sanity Check

For a real FMCG project, the first check is always date coverage, missing critical dimensions, and basic volume plausibility.

In [ ]:
summary = {
    "rows": len(sales),
    "date_min": sales["ACTUALSHIPDATE"].min(),
    "date_max": sales["ACTUALSHIPDATE"].max(),
    "categories": sales["CATEGORY"].nunique(),
    "brands": sales["BRAND"].nunique(),
    "warehouses": sales["WHSEID"].nunique(),
    "total_qty": sales["Total QTY"].sum(),
    "total_cbm": sales["Total CBM"].sum(),
}
pd.Series(summary)

In [ ]:
sales[["ACTUALSHIPDATE", "CATEGORY", "WHSEID", "BRAND", "Total QTY", "Total CBM"]].isna().sum()

## Weekly Demand Trend

The production pipeline aggregates data by Monday-start week. This demo applies the same week definition.

In [ ]:
sales["week_start"] = sales["ACTUALSHIPDATE"] - pd.to_timedelta(sales["ACTUALSHIPDATE"].dt.weekday, unit="D")
weekly = (
    sales.groupby(["week_start", "CATEGORY", "BRAND", "WHSEID"], as_index=False)
    .agg(total_qty=("Total QTY", "sum"), total_cbm=("Total CBM", "sum"))
    .sort_values("week_start")
)
weekly.head()

In [ ]:
weekly_total = weekly.groupby("week_start", as_index=False).agg(total_qty=("total_qty", "sum"), total_cbm=("total_cbm", "sum"))
px.line(weekly_total, x="week_start", y="total_qty", markers=True, title="Synthetic Weekly Quantity Trend")

## Brand and Warehouse Mix

Supply chain planners usually care about which brands drive demand and which warehouses carry CBM load.

In [ ]:
brand_mix = weekly.groupby("BRAND", as_index=False).agg(total_qty=("total_qty", "sum"), total_cbm=("total_cbm", "sum"))
px.bar(brand_mix.sort_values("total_qty", ascending=False), x="BRAND", y="total_qty", title="Synthetic Quantity by Brand")

In [ ]:
warehouse_cbm = weekly.groupby("WHSEID", as_index=False).agg(total_cbm=("total_cbm", "sum"))
px.bar(warehouse_cbm.sort_values("total_cbm", ascending=False), x="WHSEID", y="total_cbm", title="Synthetic CBM by Warehouse")

## Business Notes

- This synthetic notebook demonstrates the portfolio workflow without exposing confidential data.
- Real model training, validation, leakage-safe features, and metrics are implemented in `src/`.
- The real dashboard uses generated processed files and model predictions, which are intentionally ignored by Git.